# NLP Project — Notebook 2 : PyTorch Models
DeBERTa-v3-small + Mistral-7B QLoRA

**Lancer ce notebook dans un kernel frais, SANS avoir importe TensorFlow.**
Notebook 1 doit etre complete avant de lancer celui-ci.

---
# ============================================================
# SECTION INDEPENDANTE — NOTEBOOK 2
# POINT DE REPRISE APRES KERNEL RESTART
#
# CONDITIONS : notebook 1 complete,
# checkpoints/train_df.csv et preds_emb.npy presents
# ============================================================

In [ ]:
# Rechargement pour notebook 2 — necessite que notebook 1 soit complete
import pandas as pd, numpy as np, pickle, os, warnings, torch
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

df       = pd.read_csv('./checkpoints/df_full.csv')
train_df = pd.read_csv('./checkpoints/train_df.csv')
val_df   = pd.read_csv('./checkpoints/val_df.csv')
test_df  = pd.read_csv('./checkpoints/test_df.csv')
for _d in [df, train_df, val_df, test_df]:
    _d['avis_en']    = _d['avis_en'].fillna('').astype(str)
    _d['avis_clean'] = _d['avis_clean'].fillna('').astype(str)

y_train = np.load('./checkpoints/y_train.npy')
y_val   = np.load('./checkpoints/y_val.npy')
y_test  = np.load('./checkpoints/y_test.npy')

with open('./checkpoints/tfidf_results.pkl',   'rb') as f: tfidf_results = pickle.load(f)
with open('./checkpoints/keras_tokenizer.pkl', 'rb') as f: keras_tok     = pickle.load(f)

preds_emb  = np.load('./checkpoints/preds_emb.npy')
preds_lstm = np.load('./checkpoints/preds_lstm.npy')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1_weighted': f1}

print('Rechargement notebook 2 OK.')

In [ ]:
import sentencepiece, time
print(f'sentencepiece {sentencepiece.__version__}')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEBERTA_NAME = 'microsoft/deberta-v3-small'

t0 = time.time()
print('Loading DeBERTa-v3-small tokenizer...')
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_NAME, use_fast=False)
print(f'Tokenizer loaded in {time.time()-t0:.1f}s — {type(deberta_tokenizer).__name__}')

def hf_tokenize(batch):
    return deberta_tokenizer(
        batch['text'], truncation=True, max_length=256,
        padding=False, return_token_type_ids=False
    )

def df_to_hf_dataset(dataframe, name=''):
    t1 = time.time()
    print(f'  Tokenizing {name} ({len(dataframe)} rows)...')
    ds = Dataset.from_dict({
        'text':  dataframe['avis_en'].astype(str).tolist(),
        'label': dataframe['label'].astype(int).tolist()
    })
    ds = ds.map(hf_tokenize, batched=True, batch_size=128,
                remove_columns=['text'], desc=f'Tokenizing {name}')
    print(f'  {name} done in {time.time()-t1:.1f}s')
    return ds

t0 = time.time()
print('Building HuggingFace datasets...')
train_hf = df_to_hf_dataset(train_df, 'train')
val_hf   = df_to_hf_dataset(val_df,   'val')
test_hf  = df_to_hf_dataset(test_df,  'test')

train_hf.set_format('torch')
val_hf.set_format('torch')
test_hf.set_format('torch')

print(f'\nAll datasets ready in {time.time()-t0:.1f}s')
print(train_hf)

open('./checkpoints/_cell17a.done', 'w').close()
print('Cell 17a saved.')

---
## Cell 17b — DeBERTa-v3-small : model and trainer setup

In [ ]:
import time

accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels, average='weighted')['f1']
    return {'accuracy': acc, 'f1_weighted': f1}

t0 = time.time()
print('Loading DeBERTa-v3-small model weights...')
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_NAME,
    num_labels=3,
    id2label={0:'negative', 1:'neutral', 2:'positive'},
    label2id={'negative':0, 'neutral':1, 'positive':2},
    ignore_mismatched_sizes=True
)
deberta_model.to(device)
print(f'Model loaded in {time.time()-t0:.1f}s — {deberta_model.num_parameters():,} parameters')

deberta_args = TrainingArguments(
    output_dir='./deberta_results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    logging_dir='tensorboard_logs/deberta',
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=4,
    report_to='tensorboard',
    seed=42
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    processing_class=deberta_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=deberta_tokenizer, pad_to_multiple_of=8
    ),
    compute_metrics=compute_metrics
)
print('Trainer ready.')

open('./checkpoints/_cell17b.done', 'w').close()
print('Cell 17b saved.')

---
## Cell 17c — DeBERTa-v3-small : training, evaluation and save

In [ ]:
deberta_trainer.train()

deberta_results = deberta_trainer.predict(test_hf)
preds_deberta   = np.argmax(deberta_results.predictions, axis=-1)

print(f'\nDeBERTa-v3-small — Acc: {accuracy_score(y_test, preds_deberta):.4f} | '
      f'F1: {f1_score(y_test, preds_deberta, average="weighted"):.4f}')
print(classification_report(y_test, preds_deberta,
                             target_names=['negative','neutral','positive']))

deberta_trainer.save_model('./best_deberta_model')
deberta_tokenizer.save_pretrained('./best_deberta_model')
print('Saved to ./best_deberta_model/')

np.save('./checkpoints/preds_deberta.npy', preds_deberta)

del deberta_model, deberta_trainer
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('DeBERTa unloaded from memory')

open('./checkpoints/_cell17c.done', 'w').close()
print('Cell 17c saved.')

---
## Cell 18 — Mistral-7B with QLoRA fine-tuning

Modele : `mistralai/Mistral-7B-v0.1` (inchange)

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MISTRAL_NAME = 'mistralai/Mistral-7B-v0.1'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

mistral_tokenizer              = AutoTokenizer.from_pretrained(MISTRAL_NAME, use_fast=True)
mistral_tokenizer.pad_token    = mistral_tokenizer.eos_token
mistral_tokenizer.padding_side = 'right'

def mistral_tokenize(batch):
    return mistral_tokenizer(
        batch['text'], truncation=True, max_length=256, padding=False
    )

def df_to_mistral_dataset(dataframe):
    ds = Dataset.from_dict({
        'text':  dataframe['avis_en'].astype(str).tolist(),
        'label': dataframe['label'].astype(int).tolist()
    })
    return ds.map(mistral_tokenize, batched=True, batch_size=128,
                  remove_columns=['text'], desc='Tokenizing Mistral')

train_mistral = df_to_mistral_dataset(train_df)
val_mistral   = df_to_mistral_dataset(val_df)
test_mistral  = df_to_mistral_dataset(test_df)
train_mistral.set_format('torch')
val_mistral.set_format('torch')
test_mistral.set_format('torch')

mistral_base = AutoModelForSequenceClassification.from_pretrained(
    MISTRAL_NAME,
    num_labels=3,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    ignore_mismatched_sizes=True
)
mistral_base.config.pad_token_id = mistral_tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none'
)
mistral_model = get_peft_model(mistral_base, lora_config)
mistral_model.print_trainable_parameters()

mistral_args = TrainingArguments(
    output_dir='./mistral_results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False, bf16=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    logging_dir='tensorboard_logs/mistral',
    logging_steps=25,
    report_to='tensorboard',
    optim='paged_adamw_8bit',
    seed=42
)

mistral_trainer = Trainer(
    model=mistral_model,
    args=mistral_args,
    train_dataset=train_mistral,
    eval_dataset=val_mistral,
    tokenizer=mistral_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=mistral_tokenizer, pad_to_multiple_of=8
    ),
    compute_metrics=compute_metrics
)

mistral_trainer.train()

mistral_results = mistral_trainer.predict(test_mistral)
preds_mistral   = np.argmax(mistral_results.predictions, axis=-1)

print(f'\nMistral-7B QLoRA — Acc: {accuracy_score(y_test, preds_mistral):.4f} | '
      f'F1: {f1_score(y_test, preds_mistral, average="weighted"):.4f}')

mistral_model.save_pretrained('./best_mistral_lora')
mistral_tokenizer.save_pretrained('./best_mistral_lora')
print('Saved to ./best_mistral_lora/')

np.save('./checkpoints/preds_mistral.npy', preds_mistral)

del mistral_model, mistral_trainer, mistral_base
import gc; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Mistral unloaded from memory')

open('./checkpoints/_cell18.done', 'w').close()
print('Cell 18 saved.')

---
## Checkpoint save — run immediately after cell 18
Sauvegarde complete de tous les artefacts necessaires aux cellules 19+.

---
## Checkpoint save — preds DeBERTa et Mistral

In [ ]:
np.save('./checkpoints/preds_deberta.npy', preds_deberta)
np.save('./checkpoints/preds_mistral.npy', preds_mistral)
print('preds_deberta et preds_mistral sauvegardes.')
print('Vous pouvez maintenant lancer la comparaison finale dans notebook 1.')